# 6. Create an AI-Powered Sales Report Analyzer with LlamaIndex

In [2]:
%pip install llama-index llama-index-llms-openai-like
import os
import pandas as pd
from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.core.embeddings import MockEmbedding
from llama_index.llms.openai_like import OpenAILike
# Groq API key
os.environ["GROQ_API_KEY"] = "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"

# Sales report data
df = pd.DataFrame({
    "Month": ["Jan", "Feb", "Mar", "Apr", "May"],
    "Product": ["Laptop", "Mobile", "Tablet", "Laptop", "Mobile"],
    "Region": ["South", "North", "East", "West", "South"],
    "Revenue": [50000, 40000, 25000, 60000, 55000],
    "Profit": [10000, 8000, 4000, 12000, 11000]
})
display(df)
# Convert table into document
data_text = df.to_string(index=False)


# Use Groq with OpenAI-compatible endpoint
Settings.llm = OpenAILike(
    model="openai/gpt-oss-20b",
    api_key=os.environ["GROQ_API_KEY"],
    api_base="https://api.groq.com/openai/v1",
    is_chat_model=True,
    is_function_calling_model=False
)
# Simple embedding for classroom demo
Settings.embed_model = MockEmbedding(embed_dim=384)

document = Document(
    text=f" Sales Report:{data_text}"
)
# Create LlamaIndex index
index = VectorStoreIndex.from_documents([document])
# Create query engine
query_engine = index.as_query_engine()
# Ask question
response = query_engine.query(
    "Which product generated the highest total revenue? Show calculation."
)
print(response)

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


,Month,Product,Region,Revenue,Profit
0,Jan,Laptop,South,50000,10000
1,Feb,Mobile,North,40000,8000
2,Mar,Tablet,East,25000,4000
3,Apr,Laptop,West,60000,12000
4,May,Mobile,South,55000,11000


**Laptop** generated the highest total revenue.  

Calculation:  
- January Laptop: $50,000  
- April Laptop: $60,000  

Total Laptop revenue = $50,000 + $60,000 = **$110,000**.


# 7 Create a Market Research Agent with RAG & Cohere

In [5]:
!pip install -U cohere numpy pandas

Defaulting to user installation because normal site-packages is not writeable


In [9]:
# Install required package
%pip install -U cohere

# API Keys
import os
os.environ["COHERE_API_KEY"] = "y3roeFf01TazGMVWV2Aj8MMHmFOT2j6qq4K98EOo"

# Import Cohere
import cohere

# Create Cohere client
co = cohere.ClientV2()

# Documents for RAG
documents = [
    "Indian EV demand is led by two-wheelers and urban mobility.",
    "High battery cost and few charging stations slow EV adoption.",
    "Tata, Ola, Ather and Mahindra are major EV competitors.",
]

# User query
query = "Give a short market report on electric vehicles in India."

# Rerank documents
ranked = co.rerank(
    model="rerank-v3.5",
    query=query,
    documents=documents,
    top_n=2
)

# Select the top-ranked documents
context = "\n".join(
    documents[item.index]
    for item in ranked.results
)

# Generate final report
response = co.chat(
    model="command-a-03-2025",
    messages=[
        {
            "role": "user",
            "content": f"{query}\nUse only:\n{context}"
        }
    ],
)

# Print result
print(response.message.content[0].text)

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
**Indian Electric Vehicle Market Report**  

India’s electric vehicle (EV) market is experiencing steady growth, primarily driven by **two-wheelers and urban mobility solutions**. Two-wheelers dominate demand due to affordability, lower operational costs, and increasing consumer awareness of sustainability. Urban mobility, including electric rickshaws and ride-sharing services, is also gaining traction as cities push for cleaner transportation options.  

Key players in the Indian EV space include **Tata Motors**, **Ola Electric**, **Ather Energy**, and **Mahindra & Mahindra**. Tata Motors leads the four-wheeler segment with models like the Nexon EV, while Ola Electric dominates the two-wheeler market with its popular electric scooters. Ather Energy has carved a niche in premium electric scooters, and Mahindra continues to expand its EV portfol

# 8

In [8]:
%pip install -U phidata
import os
import pandas as pd

from phi.agent import Agent
from phi.model.groq import Groq

# Use your NEW Groq API key
os.environ["GROQ_API_KEY"] = "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"

# Sample sales data
df = pd.DataFrame({
    "Month": ["Jan", "Feb", "Mar", "Apr", "May"],
    "Product": ["Laptop", "Mobile", "Tablet", "Laptop", "Mobile"],
    "Region": ["South", "North", "East", "West", "South"],
    "Revenue": [50000, 40000, 25000, 60000, 55000],
    "Profit": [10000, 8000, 4000, 12000, 11000]
})

display(df)

# Convert dataframe to text
data_text = df.to_string(index=False)

# Create Phidata agent
data_agent = Agent(
    name="Data Analysis Agent",
    model=Groq(id="openai/gpt-oss-120b"),
    instructions=[
        "You are a data analysis assistant.",
        "Analyze the given sales data carefully.",
        "Show calculations clearly.",
        "Give short business insights."
    ],
    markdown=True
)

# Question
question = "Which product has the highest total revenue? Show calculation."

# Run agent
response = data_agent.run(
    f"""
Here is the sales data:

{data_text}

Question:
{question}
"""
)

print(response.content)

,Month,Product,Region,Revenue,Profit
0,Jan,Laptop,South,50000,10000
1,Feb,Mobile,North,40000,8000
2,Mar,Tablet,East,25000,4000
3,Apr,Laptop,West,60000,12000
4,May,Mobile,South,55000,11000


**Total Revenue by Product**

| Product | Revenue (Jan) | Revenue (Apr) | Revenue (Feb) | Revenue (May) | Revenue (Mar) | **Total Revenue** |
|---------|--------------|--------------|--------------|--------------|--------------|-------------------|
| Laptop  | 50,000 | 60,000 | – | – | – | **110,000** |
| Mobile  | – | – | 40,000 | 55,000 | – | **95,000** |
| Tablet  | – | – | – | – | 25,000 | **25,000** |

**Calculation**

- **Laptop:** 50,000 (Jan) + 60,000 (Apr) = **110,000**  
- **Mobile:** 40,000 (Feb) + 55,000 (May) = **95,000**  
- **Tablet:** 25,000 (Mar) = **25,000**  

**Answer**

The **Laptop** product has the highest total revenue, amounting to **$110,000**.  

**Brief Insight**

- Laptops generate the most revenue despite being sold in only two months, indicating a strong market demand or higher price point.
- Mobile devices are close behind; focusing on cross‑selling or bundling could capture additional share.
- Tablet sales are comparatively low, suggesting an opportunit

In [8]:
import pandas as pd
import os
from phi.agent import Agent
from phi.model.groq import Groq

os.environ["GROQ_API_KEY"]="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"

df = pd.DataFrame({
    "Month":["January","February","March","April","May"],
    "Product":["Laptop","Mobile","Tablet","Laptop","Mobile"],
    "Region":["North","South","East","West","South"],
    "Revenue":[50000,40000,25000,60000,55000],
    "Profit":[8000,10000,6000,12000,15000]
})

print(df)

data_text = df.to_string(index=False)

agent = Agent(
    name = "Data Analysis Agent",
    model = Groq(id="openai/gpt-oss-20b"),
    instructions=[
        "You are a Data Analysis Agent",
        "you have to carefully analyze the data provided",
        "You have to interpret the data correctly and provide reasoning for the decisions you make",
        "Make proper calculations"
    ]
)

question = "which product has the highest total revenu and which region has the highest profit"

response = agent.run(f" Question = {question} Data : {data_text}")

print(response.content)

      Month Product Region  Revenue  Profit
0   January  Laptop  North    50000    8000
1  February  Mobile  South    40000   10000
2     March  Tablet   East    25000    6000
3     April  Laptop   West    60000   12000
4       May  Mobile  South    55000   15000
**Step‑by‑step calculation**

| Month   | Product | Region | Revenue | Profit |
|---------|---------|--------|---------|--------|
| January | Laptop  | North  | 50,000  | 8,000  |
| February| Mobile  | South  | 40,000  | 10,000 |
| March   | Tablet  | East   | 25,000  | 6,000  |
| April   | Laptop  | West   | 60,000  | 12,000 |
| May     | Mobile  | South  | 55,000  | 15,000 |

---

### 1. Highest total revenue by product

Add the revenue values for each product:

| Product | Sum of Revenue |
|---------|----------------|
| Laptop  | 50,000 + 60,000 = **110,000** |
| Mobile  | 40,000 + 55,000 = 95,000 |
| Tablet  | 25,000 |

**Result:** The product with the highest total revenue is **Laptop** (110,000).

---

### 2. Highest pro